# TinyML : Apprentissage Automatique pour Systemes Embarques

## Contexte et Objectifs

Ce notebook est une introduction pratique au TinyML, un domaine de l'apprentissage automatique qui se concentre sur le deploiement de modeles sur des appareils a tres faibles ressources, comme les microcontroleurs. L'objectif est de montrer le flux de travail complet, de la formation d'un modele a sa conversion et sa quantification pour une utilisation sur des systemes embarques.

Comme cas d'utilisation, nous allons entrainer un modele de detection de mots-cles audio (`Keyword Spotting`), une tache courante en TinyML.

### Flux de Travail Aborde :

1.  **Introduction au TinyML :** Pourquoi est-il important d'executer des modeles a la peripherie (`Edge`) et quels sont les defis associes aux appareils a ressources limitees ?
2.  **Jeu de Donnees Audio :** Nous utiliserons un jeu de donnees de commandes vocales simples pour entrainer notre modele.
3.  **Pretraitement des Donnees Audio :** Transformation des fichiers audio bruts en spectrogrammes, une representation visuelle du spectre de frequence du son, plus facile a traiter par un modele de type CNN.
4.  **Formation d'un Modele CNN :** Nous entrainons un reseau de neurones a convolution simple avec TensorFlow/Keras pour classifier les commandes vocales.
5.  **Conversion vers TensorFlow Lite :** La premiere etape de l'optimisation, ou nous convertissons le modele Keras au format TensorFlow Lite (TFLite), concu pour les appareils mobiles et embarques.
6.  **Quantification :** Nous appliquons la quantification post-entrainement pour reduire drastiquement la taille du modele et le rendre compatible avec les microcontroleurs, en convertissant les poids de nombres a virgule flottante en entiers 8 bits.
7.  **Inference avec l'Interpreteur TFLite :** Nous simulons l'execution du modele quantifie sur un appareil embarque en utilisant l'interpreteur TFLite pour faire des predictions.

_Derniere mise a jour : 2026-02-16_

In [1]:
# --- 1. Installation des Dependances ---
%pip install -q tensorflow numpy matplotlib
print("Dependances installees.")

SyntaxError: invalid syntax (2051176013.py, line 1)

In [2]:
# --- 2. Imports et Configuration ---
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SyntaxError: invalid syntax (3714364521.py, line 1)

## 3. Telechargement et Preparation du Jeu de Donnees

Nous telechargeons le jeu de donnees "Speech Commands", qui contient de courts extraits audio de commandes. Nous nous concentrerons sur la detection de deux mots : `yes` et `no`, et une categorie pour les sons inconnus.

In [3]:
# Telecharger et extraire le jeu de donnees
data_dir = tf.keras.utils.get_file(
    'mini_speech_commands.zip',
    origin="http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip",
    extract=True,
    cache_dir='.', cache_subdir='data'
)
data_dir = os.path.join(os.path.dirname(data_dir), 'mini_speech_commands')

# Obtenir la liste des commandes (etiquettes)
labels = np.array(tf.io.gfile.listdir(str(data_dir)))
labels = labels[labels != 'README.md']
logger.info(f"Commandes detectees : {labels}")

# Charger les chemins des fichiers audio
filenames = tf.io.gfile.glob(str(data_dir) + '/*/*')
filenames = tf.random.shuffle(filenames)
num_samples = len(filenames)
logger.info(f"Nombre total d'echantillons : {num_samples}")

SyntaxError: invalid syntax (823652641.py, line 1)

## 4. Pretraitement : Audio vers Spectrogrammes

Les modeles d'apprentissage automatique fonctionnent mieux avec des donnees structurees. Nous allons donc convertir les formes d'onde audio en spectrogrammes, qui sont des images representant l'intensite du son a differentes frequences au fil du temps.

In [4]:
# Diviser les donnees en ensembles d'entrainement, de validation et de test
train_files = filenames[:int(num_samples * 0.8)]
val_files = filenames[int(num_samples * 0.8): int(num_samples * 0.9)]
test_files = filenames[-int(num_samples * 0.1):]

def decode_audio(audio_binary):
    audio, _ = tf.audio.decode_wav(audio_binary)
    return tf.squeeze(audio, axis=-1)

def get_label(file_path):
    parts = tf.strings.split(file_path, os.path.sep)
    return parts[-2] == labels

def get_waveform_and_label(file_path):
    label = get_label(file_path)
    audio_binary = tf.io.read_file(file_path)
    waveform = decode_audio(audio_binary)
    return waveform, label

def get_spectrogram(waveform):
    # Zero-padding pour avoir des entrees de meme longueur (1 seconde a 16000Hz)
    zero_padding = tf.zeros([16000] - tf.shape(waveform), dtype=tf.float32)
    waveform = tf.cast(waveform, tf.float32)
    equal_length = tf.concat([waveform, zero_padding], 0)
    spectrogram = tf.signal.stft(equal_length, frame_length=255, frame_step=128)
    spectrogram = tf.abs(spectrogram)
    return spectrogram

def get_spectrogram_and_label_id(audio, label):
    spectrogram = get_spectrogram(audio)
    label_id = tf.argmax(label).numpy()
    return spectrogram, label_id

# Creer le jeu de donnees TensorFlow
AUTOTUNE = tf.data.AUTOTUNE
def preprocess_dataset(files):
    files_ds = tf.data.Dataset.from_tensor_slices(files)
    output_ds = files_ds.map(get_waveform_and_label, num_parallel_calls=AUTOTUNE)
    output_ds = output_ds.map(get_spectrogram_and_label_id, num_parallel_calls=AUTOTUNE)
    return output_ds

train_ds = preprocess_dataset(train_files)
val_ds = preprocess_dataset(val_files)
test_ds = preprocess_dataset(test_files)

SyntaxError: invalid syntax (3041504807.py, line 1)

## 5. Entrainement du Modele CNN

In [5]:
batch_ds = train_ds.batch(32)
val_batch_ds = val_ds.batch(32)

for spectrogram, _ in train_ds.take(1):
    input_shape = spectrogram.shape
logger.info(f'Forme de l'entree : {input_shape}')

# Definition du modele
model = models.Sequential([
    layers.Input(shape=input_shape),
    layers.Resizing(32, 32), # Redimensionner pour uniformiser
    layers.Conv2D(32, 3, activation='relu'),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(labels)),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

logger.info("Debut de l'entrainement du modele...")
history = model.fit(
    batch_ds,
    validation_data=val_batch_ds,
    epochs=10
)

SyntaxError: invalid syntax (416476538.py, line 1)

## 6. Conversion en TensorFlow Lite et Quantification

In [ ]:
# Conversion du modele Keras en modele TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Quantification post-entrainement (INT8)
def representative_dataset():
    for data, _ in train_ds.take(100):
        yield [data[tf.newaxis, ...]]

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_quant = converter.convert()

# Sauvegarder les modeles
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)
with open('model_quant.tflite', 'wb') as f:
    f.write(tflite_model_quant)

logger.info(f"Taille du modele TFLite original : {len(tflite_model) / 1024:.2f} KB")
logger.info(f"Taille du modele TFLite quantifie : {len(tflite_model_quant) / 1024:.2f} KB")

## 7. Simulation de l'Inference sur le Bord de Reseau (Edge)

In [7]:
# Test avec l'interpreteur TFLite
interpreter = tf.lite.Interpreter(model_content=tflite_model_quant)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

# Faire une prediction sur un echantillon de test
for test_spectrogram, test_label in test_ds.take(1): pass

# Si le modele quantifie attend des entiers, il faut quantifier l'entree
if input_details['dtype'] == np.int8:
    input_scale, input_zero_point = input_details["quantization"]
    test_spectrogram = test_spectrogram.numpy().astype(np.float32) / input_scale + input_zero_point
    test_spectrogram = test_spectrogram.astype(np.int8)

interpreter.set_tensor(input_details["index"], test_spectrogram[np.newaxis, ...])
interpreter.invoke()
output_data = interpreter.get_tensor(output_details["index"])

predicted_label_id = np.argmax(output_data)
logger.info(f"Prediction : {labels[predicted_label_id]}")
logger.info(f"Vraie etiquette : {labels[test_label]}")

Notebook executed (marker) — 2026-02-16 00:44:24
